# 🔥 Deteksi Kebakaran Hutan – Provinsi Riau
## Google Earth Engine via Python API (geemap + ee)
**Periode:** Januari 2015 – Desember 2025  
**Variabel:** MODIS Fire (MCD64A1, MOD14A1, MYD14A1), LST, CHIRPS Rainfall, NDVI, ERA5 Wind  
**Output:** Statistik CSV bulanan + Peta interaktif + Ekspor ke Google Drive

---
> **Cara pakai:**  
> 1. Jalankan sel **Instalasi** terlebih dahulu  
> 2. **Autentikasi** akun Google Earth Engine  
> 3. Jalankan sel satu per satu dari atas ke bawah  
> 4. Hasil ekspor CSV & GeoTIFF tersimpan di Google Drive folder `GEE_Fire_Riau`


In [ ]:
# ============================================================
# 1. INSTALASI LIBRARY
# ============================================================
!pip install earthengine-api geemap folium matplotlib pandas --quiet
print("✅ Instalasi selesai")


In [ ]:
# ============================================================
# 2. IMPORT LIBRARY & AUTENTIKASI GEE
# ============================================================
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Autentikasi Google Earth Engine (muncul popup login browser)
ee.Authenticate()
ee.Initialize(project='your-gee-project-id')   # ← ganti dengan Project ID GEE Anda
print("✅ Google Earth Engine berhasil diinisialisasi")


---
## 📍 3. Definisi Wilayah Studi (ROI)


In [ ]:
# ============================================================
# 3. DEFINISI WILAYAH STUDI
# ============================================================

# Batas Provinsi Riau (Level 1)
riau = (ee.FeatureCollection('FAO/GAUL/2015/level1')
        .filter(ee.Filter.And(
            ee.Filter.eq('ADM0_NAME', 'Indonesia'),
            ee.Filter.eq('ADM1_NAME', 'Riau')
        )))

# Batas Kabupaten (Level 2)
riau_kab = (ee.FeatureCollection('FAO/GAUL/2015/level2')
            .filter(ee.Filter.And(
                ee.Filter.eq('ADM0_NAME', 'Indonesia'),
                ee.Filter.eq('ADM1_NAME', 'Riau')
            )))

print("✅ ROI Provinsi Riau & Kabupaten berhasil didefinisikan")
print(f"   Jumlah kabupaten/kota: {riau_kab.size().getInfo()}")


---
## ⚙️ 4. Fungsi Pemrosesan Kebakaran Per Bulan
Fungsi `process_fire_month()` menggabungkan 6 sumber data satelit:
- **MODIS/061/MCD64A1** – Area terbakar (burned area)
- **MODIS/061/MOD14A1 / MODIS/061/MYD14A1** – Api aktif Terra & Aqua
- **MODIS/061/MOD11A2** – Land Surface Temperature (LST)
- **CHIRPS Daily** – Curah hujan bulanan
- **MODIS/061/MOD13A2** – NDVI (kondisi vegetasi)
- **ERA5-Land Monthly Aggregated** – Kecepatan angin


In [ ]:
# ============================================================
# 4. FUNGSI DETEKSI KEBAKARAN PER BULAN
# ============================================================

def process_fire_month(year: int, month: int) -> ee.Feature:
    """
    Memproses data kebakaran untuk satu bulan tertentu.

    Parameter
    ---------
    year  : int – Tahun (2015–2025)
    month : int – Bulan (1–12)

    Returns
    -------
    ee.Feature – Berisi statistik kebakaran bulan tersebut
    """
    start_date = ee.Date.fromYMD(year, month, 1)
    end_date   = start_date.advance(1, 'month')

    # ── 4a. MODIS Burned Area (MCD64A1) ──────────────────────
    burned_col = (ee.ImageCollection('MODIS/061/MCD64A1')
                  .filterDate(start_date, end_date)
                  .filterBounds(riau)
                  .select('BurnDate'))

    fire_mask_mcd64 = ee.Algorithms.If(
        burned_col.size().gt(0),
        burned_col.first().gt(0).rename('fire_mcd64'),
        ee.Image(0).rename('fire_mcd64')
    )
    fire_mask_mcd64 = ee.Image(fire_mask_mcd64)

    # ── 4b. MODIS Terra Active Fire (MOD14A1) ────────────────
    mod_fire_col = (ee.ImageCollection('MODIS/061/MOD14A1')
                    .filterDate(start_date, end_date)
                    .filterBounds(riau)
                    .select('FireMask'))
    fire_mask_mod14 = ee.Image(ee.Algorithms.If(
        mod_fire_col.size().gt(0),
        mod_fire_col.max().gte(8),
        ee.Image(0)
    )).rename('fire_mod14')

    # ── 4c. MODIS Aqua Active Fire (MYD14A1) ─────────────────
    myd_fire_col = (ee.ImageCollection('MODIS/061/MYD14A1')
                    .filterDate(start_date, end_date)
                    .filterBounds(riau)
                    .select('FireMask'))
    fire_mask_myd14 = ee.Image(ee.Algorithms.If(
        myd_fire_col.size().gt(0),
        myd_fire_col.max().gte(8),
        ee.Image(0)
    )).rename('fire_myd14')

    # ── 4d. Fire Mask Gabungan ────────────────────────────────
    # Terdeteksi jika ≥ 1 sensor mendeteksi api
    combined_fire = (fire_mask_mcd64
                     .add(fire_mask_mod14)
                     .add(fire_mask_myd14)
                     .gt(0)
                     .rename('fire_combined')
                     .clip(riau))

    # ── 4e. Land Surface Temperature (MOD11A2) ───────────────
    empty_sensor = ee.Image.constant(0).updateMask(ee.Image.constant(0))
    lst_col = (ee.ImageCollection('MODIS/061/MOD11A2')
               .filterDate(start_date, end_date)
               .filterBounds(riau)
               .select('LST_Day_1km'))
    lst = ee.Image(ee.Algorithms.If(
        lst_col.size().gt(0),
        lst_col.mean().multiply(0.02).subtract(273.15).rename('LST_celsius'),
        empty_sensor.rename('LST_celsius')
    ))

    # ── 4f. Curah Hujan (CHIRPS) ──────────────────────────────
    rainfall = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                .filterDate(start_date, end_date)
                .filterBounds(riau)
                .select('precipitation')
                .sum()              # Total bulanan (mm)
                .rename('rainfall_mm'))

    # ── 4g. NDVI (Kondisi Vegetasi) ───────────────────────────
    ndvi_col = (ee.ImageCollection('MODIS/061/MOD13A2')
                .filterDate(start_date, end_date)
                .filterBounds(riau)
                .select('NDVI'))
    ndvi = ee.Image(ee.Algorithms.If(
        ndvi_col.size().gt(0),
        ndvi_col.mean().multiply(0.0001).rename('NDVI'),
        empty_sensor.rename('NDVI')
    ))

    # ── 4h. Kecepatan Angin (ERA5) ────────────────────────────
    era5_col = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                .filterDate(start_date, end_date)
                .filterBounds(riau))

    wind_speed = ee.Image(ee.Algorithms.If(
        era5_col.size().gt(0),
        era5_col.first().expression(
            'sqrt(u**2 + v**2)',
            {
                'u': era5_col.first().select('u_component_of_wind_10m'),
                'v': era5_col.first().select('v_component_of_wind_10m')
            }
        ).rename('wind_speed'),
        empty_sensor.rename('wind_speed')
    ))

    # ── 4i. Fire Risk Index (FRI) ─────────────────────────────
    # FRI = LST_norm*0.35 + (1−NDVI_norm)*0.35 + (1−rain_norm)*0.20 + wind_norm*0.10
    lst_norm  = lst.subtract(20).divide(20).clamp(0, 1)
    ndvi_norm = ndvi.clamp(-0.2, 0.9).add(0.2).divide(1.1)
    rain_norm = rainfall.divide(300).clamp(0, 1)
    wind_norm = wind_speed.divide(10).clamp(0, 1)

    fire_risk_index = (lst_norm.multiply(0.35)
                       .add(ndvi_norm.subtract(1).abs().multiply(0.35))
                       .add(rain_norm.subtract(1).abs().multiply(0.20))
                       .add(wind_norm.multiply(0.10))
                       .rename('fire_risk_index')
                       .clip(riau))

    # ── 4j. Hitung Statistik Reduksi ─────────────────────────
    pixel_area = ee.Image.pixelArea().divide(10000)   # m² → ha
    fire_area  = combined_fire.multiply(pixel_area)

    fire_stats    = fire_area.reduceRegion(
        reducer=ee.Reducer.sum(), geometry=riau.geometry(),
        scale=500, maxPixels=1e13)

    lst_stats     = lst.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=riau.geometry(),
        scale=1000, maxPixels=1e13)

    rain_stats    = rainfall.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=riau.geometry(),
        scale=5000, maxPixels=1e13)

    ndvi_stats    = ndvi.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=riau.geometry(),
        scale=1000, maxPixels=1e13)

    wind_stats    = wind_speed.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=riau.geometry(),
        scale=25000, maxPixels=1e13)

    fri_stats     = fire_risk_index.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=riau.geometry(),
        scale=1000, maxPixels=1e13)

    lst_count     = lst.reduceRegion(
        reducer=ee.Reducer.count(), geometry=riau.geometry(),
        scale=1000, maxPixels=1e13)
    ndvi_count    = ndvi.reduceRegion(
        reducer=ee.Reducer.count(), geometry=riau.geometry(),
        scale=1000, maxPixels=1e13)
    wind_count    = wind_speed.reduceRegion(
        reducer=ee.Reducer.count(), geometry=riau.geometry(),
        scale=25000, maxPixels=1e13)

    return ee.Feature(None, {
        'year':             year,
        'month':            month,
        'date_label':       ee.String(ee.Number(year).format('%d')).cat('-')
                            .cat(ee.String(ee.Number(month).format('%02d'))),
        'fire_area_ha':     fire_stats.get('fire_combined'),
        'mean_lst_celsius': lst_stats.get('LST_celsius'),
        'total_rain_mm':    rain_stats.get('rainfall_mm'),
        'mean_ndvi':        ndvi_stats.get('NDVI'),
        'mean_wind_ms':     wind_stats.get('wind_speed'),
        'fire_risk_index':  fri_stats.get('fire_risk_index'),
        'has_fire_event':   fire_stats.getNumber('fire_combined').gt(100),
        'lst_valid_pixels': lst_count.get('LST_celsius'),
        'ndvi_valid_pixels': ndvi_count.get('NDVI'),
        'wind_valid_pixels': wind_count.get('wind_speed'),
        'modis_collection_version': 'MODIS/061',
        'source_modis_collection': 'MODIS/061',
        'source_modis061_verified': 1
    })

print("✅ Fungsi process_fire_month() berhasil didefinisikan")


# ============================================================
# 5. ITERASI SEMUA BULAN & DOWNLOAD CSV LANGSUNG
# ============================================================

years  = list(range(2015, 2026))   # 2015–2025
months = list(range(1, 13))        # Januari–Desember

all_features = []
for y in years:
    for m in months:
        all_features.append(process_fire_month(y, m))

stats_fc = ee.FeatureCollection(all_features)
print(f"✅ FeatureCollection dibuat: {len(all_features)} bulan diproses")
print("⏳ Mengambil data dari GEE (proses ini memakan waktu beberapa menit)...")

# ── Ambil data langsung ke Python (tanpa Google Drive) ───────────────────────
data = stats_fc.getInfo()  # dict dengan key 'features'

rows = []
for feat in data['features']:
    props = feat['properties']
    rows.append({
        'year':             props.get('year'),
        'month':            props.get('month'),
        'date_label':       props.get('date_label'),
        'fire_area_ha':     props.get('fire_area_ha'),
        'mean_lst_celsius': props.get('mean_lst_celsius'),
        'total_rain_mm':    props.get('total_rain_mm'),
        'mean_ndvi':        props.get('mean_ndvi'),
        'mean_wind_ms':     props.get('mean_wind_ms'),
        'fire_risk_index':  props.get('fire_risk_index'),
        'has_fire_event':   props.get('has_fire_event'),
        'lst_valid_pixels': props.get('lst_valid_pixels'),
        'ndvi_valid_pixels': props.get('ndvi_valid_pixels'),
        'wind_valid_pixels': props.get('wind_valid_pixels'),
        'modis_collection_version': props.get('modis_collection_version'),
        'source_modis_collection': props.get('source_modis_collection'),
        'source_modis061_verified': props.get('source_modis061_verified'),
    })

df_fire = pd.DataFrame(rows).sort_values(['year','month']).reset_index(drop=True)
df_fire['date'] = pd.to_datetime(df_fire[['year','month']].assign(day=1))

print(f"✅ Data berhasil dimuat: {len(df_fire)} baris")
print(df_fire.head(10).to_string(index=False))


In [ ]:
# ============================================================
# 5b. DOWNLOAD CSV KE KOMPUTER LOKAL
# ============================================================
from google.colab import files

csv_path = 'riau_fire_monthly_stats_modis061.csv'
df_fire.to_csv(csv_path, index=False)
print(f"✅ CSV disimpan: {csv_path}  ({len(df_fire)} baris × {len(df_fire.columns)} kolom)")

files.download(csv_path)   # browser otomatis mengunduh file


In [ ]:
# ============================================================
# 5c. CEK DATAFRAME (opsional)
# ============================================================
print("=== Info DataFrame ===")
print(df_fire.info())
print()
print("=== Statistik Deskriptif ===")
print(df_fire.describe().round(3))


In [ ]:
# ============================================================
# 5b. MONITOR STATUS TASK (opsional, jalankan berkala)
# ============================================================
import time

def monitor_task(task, interval_sec=30, max_wait_min=30):
    """Monitor task GEE hingga selesai atau timeout."""
    print(f"⏳ Monitoring task: {task.id}")
    for _ in range(int(max_wait_min * 60 / interval_sec)):
        status = task.status()
        state  = status['state']
        print(f"   Status: {state}")
        if state in ('COMPLETED', 'FAILED', 'CANCELLED'):
            print(f"✅ Task selesai dengan status: {state}")
            return status
        time.sleep(interval_sec)
    print("⚠️ Timeout monitoring. Task masih berjalan.")
    return None

# Uncomment untuk mulai monitoring:
# monitor_task(task_csv)


---
## 🗺️ 6. Visualisasi Peta Interaktif – September 2015 (El Niño)


In [ ]:
# ============================================================
# 6. VISUALISASI PETA – September 2015 (Puncak El Niño)
# ============================================================

ex_year, ex_month = 2015, 9
ex_start = ee.Date.fromYMD(ex_year, ex_month, 1)
ex_end   = ex_start.advance(1, 'month')

# ── Data ──────────────────────────────────────────────────
ex_burned = (ee.ImageCollection('MODIS/061/MCD64A1')
             .filterDate(ex_start, ex_end)
             .filterBounds(riau)
             .select('BurnDate')
             .first()
             .gt(0)
             .clip(riau))

ex_mod14  = (ee.ImageCollection('MODIS/061/MOD14A1')
             .filterDate(ex_start, ex_end)
             .filterBounds(riau)
             .select('FireMask')
             .max()
             .gte(8)
             .clip(riau))

ex_lst    = (ee.ImageCollection('MODIS/061/MOD11A2')
             .filterDate(ex_start, ex_end)
             .filterBounds(riau)
             .select('LST_Day_1km')
             .mean()
             .multiply(0.02).subtract(273.15)
             .clip(riau))

ex_ndvi   = (ee.ImageCollection('MODIS/061/MOD13A2')
             .filterDate(ex_start, ex_end)
             .filterBounds(riau)
             .select('NDVI')
             .mean()
             .multiply(0.0001)
             .clip(riau))

ex_rain   = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
             .filterDate(ex_start, ex_end)
             .filterBounds(riau)
             .select('precipitation')
             .sum()
             .clip(riau))

# ── Vis Params ────────────────────────────────────────────
vis = {
    'fire_mask':   {'min': 0, 'max': 1, 'palette': ['00000000', 'FF0000']},
    'active_fire': {'min': 0, 'max': 1, 'palette': ['00000000', 'FF6600']},
    'lst':         {'min': 22, 'max': 45,
                    'palette': ['blue','cyan','yellow','orange','red','darkred']},
    'ndvi':        {'min': -0.1, 'max': 0.8,
                    'palette': ['8B4513','FFFF00','00FF00','006400']},
    'rainfall':    {'min': 0, 'max': 400,
                    'palette': ['ffffff','lightblue','0000ff','000080']},
}

# ── Peta Interaktif (geemap) ──────────────────────────────
Map = geemap.Map(center=[0.5, 102.0], zoom=8)
Map.add_basemap('HYBRID')

Map.addLayer(ex_lst,    vis['lst'],         'LST Sept 2015 (°C)')
Map.addLayer(ex_ndvi,   vis['ndvi'],        'NDVI Sept 2015')
Map.addLayer(ex_rain,   vis['rainfall'],    'Curah Hujan Sept 2015 (mm)')
Map.addLayer(ex_mod14,  vis['active_fire'], 'Api Aktif MOD14 Sept 2015')
Map.addLayer(ex_burned, vis['fire_mask'],   'Area Terbakar MCD64 Sept 2015')
Map.addLayer(riau,      {'color': 'yellow', 'fillColor': '00000000'}, 'Batas Riau')

# Legenda
legend_dict = {
    'Area Terbakar (MCD64)':   '#FF0000',
    'Api Aktif (MOD14)':       '#FF6600',
    'Risiko Sangat Tinggi':    '#FF0080',
    'Risiko Tinggi':           '#FF8800',
    'Risiko Sedang':           '#FFFF00',
    'Risiko Rendah':           '#00CC44',
}
Map.add_legend(title='Legenda Fire Mask', legend_dict=legend_dict,
               position='bottomleft')

print("🗺️ Peta interaktif siap. Tampil di bawah sel ini.")
Map


---
## 💾 7. Ekspor Fire Mask sebagai GeoTIFF ke Google Drive


In [ ]:
# ============================================================
# 7. EKSPOR FIRE MASK IMAGE → GOOGLE DRIVE
# ============================================================
# Catatan: untuk data raster (GeoTIFF) tetap menggunakan Google Drive
# karena ukuran file bisa sangat besar (ratusan MB).
# Alternatif: gunakan geemap.ee_export_image() untuk download langsung
# jika area/resolusi tidak terlalu besar.

# ── Opsi A: Download langsung via geemap (untuk area kecil) ──────────────────
# geemap.ee_export_image(
#     ex_burned.toFloat(),
#     filename='riau_firemask_2015_09.tif',
#     scale=500,
#     region=riau.geometry(),
#     file_per_band=False
# )

# ── Opsi B: Export ke Google Drive (untuk area/resolusi besar) ───────────────
task_img = ee.batch.Export.image.toDrive(
    image=ex_burned.toFloat(),
    description='Riau_FireMask_Sept2015',
    folder='GEE_Fire_Riau',
    fileNamePrefix='riau_firemask_2015_09',
    region=riau.geometry(),
    scale=500,
    crs='EPSG:4326',
    maxPixels=1e13
)
task_img.start()
print("🚀 Task GeoTIFF ke Google Drive dimulai.")
print("   Atau uncomment Opsi A di atas untuk download langsung.")


---
## 🏘️ 8. Statistik Kebakaran Per Kabupaten – September 2015


In [ ]:
# ============================================================
# 8. STATISTIK KEBAKARAN PER KABUPATEN – DOWNLOAD LANGSUNG
# ============================================================
from google.colab import files

kab_stats = ex_burned.reduceRegions(
    collection=riau_kab,
    reducer=ee.Reducer.sum().setOutputs(['fire_pixels']),
    scale=500
)

def add_area(f):
    pix  = f.getNumber('fire_pixels')
    area = pix.multiply(500).multiply(500).divide(10000)  # 500m piksel → ha
    return f.set('fire_area_ha', area).set('year', ex_year).set('month', ex_month)

kab_stats_area = kab_stats.map(add_area)

print("⏳ Mengambil data per kabupaten dari GEE...")
kab_data = kab_stats_area.select(['ADM2_NAME','fire_area_ha','year','month']).getInfo()

kab_rows = [feat['properties'] for feat in kab_data['features']]
df_kab = pd.DataFrame(kab_rows).sort_values('fire_area_ha', ascending=False).reset_index(drop=True)

print(f"✅ Data kabupaten berhasil dimuat: {len(df_kab)} kabupaten/kota")
print(df_kab.to_string(index=False))

# ── Download CSV ───────────────────────────────────────────
kab_csv = 'riau_fire_kabupaten_2015_09.csv'
df_kab.to_csv(kab_csv, index=False)
files.download(kab_csv)
print(f"\n✅ CSV per kabupaten diunduh: {kab_csv}")


---
## 📈 9. Visualisasi Data CSV (setelah ekspor selesai)
Setelah file CSV ter-download dari Google Drive, jalankan sel berikut untuk membuat grafik tren kebakaran.


In [ ]:
# ============================================================
# 9. PLOT TREN KEBAKARAN (jalankan setelah CSV diunduh)
# ============================================================
# Ganti path di bawah dengan lokasi file CSV Anda:
# df_fire sudah tersedia dari sel No.5 di atas — langsung digunakan

try:
    df = df_fire.copy()  # gunakan DataFrame yang sudah ada
    df['date'] = pd.to_datetime(df[['year','month']].assign(day=1))
    df = df.sort_values('date').reset_index(drop=True)

    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle('Tren Kebakaran Hutan – Provinsi Riau (2015–2025)',
                 fontsize=16, fontweight='bold', y=0.98)

    # 1. Luas Terbakar
    axes[0,0].fill_between(df['date'], df['fire_area_ha'] / 1000,
                            alpha=0.7, color='crimson')
    axes[0,0].plot(df['date'], df['fire_area_ha'] / 1000, color='darkred', lw=1)
    axes[0,0].set_title('Luas Area Terbakar (ribu ha)')
    axes[0,0].set_ylabel('1,000 ha')
    axes[0,0].grid(alpha=0.3)

    # 2. Suhu Permukaan (LST)
    axes[0,1].plot(df['date'], df['mean_lst_celsius'], color='orange', lw=1.5)
    axes[0,1].axhline(40, color='red', ls='--', alpha=0.7, label='Ambang kritis 40°C')
    axes[0,1].set_title('Suhu Permukaan Rata-rata (LST)')
    axes[0,1].set_ylabel('°C')
    axes[0,1].legend(fontsize=9)
    axes[0,1].grid(alpha=0.3)

    # 3. Curah Hujan
    axes[1,0].bar(df['date'], df['total_rain_mm'], width=25,
                  color='steelblue', alpha=0.8)
    axes[1,0].axhline(100, color='red', ls='--', alpha=0.7, label='Batas kering 100mm')
    axes[1,0].set_title('Curah Hujan Bulanan (CHIRPS)')
    axes[1,0].set_ylabel('mm')
    axes[1,0].legend(fontsize=9)
    axes[1,0].grid(alpha=0.3)

    # 4. NDVI
    axes[1,1].plot(df['date'], df['mean_ndvi'], color='forestgreen', lw=1.5)
    axes[1,1].set_title('NDVI Rata-rata (Kondisi Vegetasi)')
    axes[1,1].set_ylabel('NDVI')
    axes[1,1].grid(alpha=0.3)

    # 5. Kecepatan Angin
    axes[2,0].plot(df['date'], df['mean_wind_ms'], color='teal', lw=1.5)
    axes[2,0].set_title('Kecepatan Angin (ERA5)')
    axes[2,0].set_ylabel('m/s')
    axes[2,0].grid(alpha=0.3)

    # 6. Fire Risk Index
    axes[2,1].fill_between(df['date'], df['fire_risk_index'],
                            alpha=0.7, color='darkorange')
    axes[2,1].plot(df['date'], df['fire_risk_index'], color='saddlebrown', lw=1)
    axes[2,1].axhline(0.5, color='red', ls='--', alpha=0.7, label='Risiko Tinggi (0.5)')
    axes[2,1].axhline(0.7, color='darkred', ls='--', alpha=0.7, label='Risiko Sangat Tinggi (0.7)')
    axes[2,1].set_title('Fire Risk Index (FRI)')
    axes[2,1].set_ylabel('Indeks (0–1)')
    axes[2,1].legend(fontsize=9)
    axes[2,1].grid(alpha=0.3)

    for ax in axes.flat:
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.savefig('riau_fire_trend_2015_2025.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Grafik disimpan sebagai 'riau_fire_trend_2015_2025.png'")

except NameError:
    print("⚠️  Jalankan sel No.5 terlebih dahulu untuk memuat data.")


In [ ]:
# ============================================================
# 10. RINGKASAN STATISTIK & TAHUN PALING PARAH
# ============================================================
try:
    # Ringkasan tahunan
    annual = (df.groupby('year')
                .agg(
                    total_fire_ha=('fire_area_ha', 'sum'),
                    max_fire_ha  =('fire_area_ha', 'max'),
                    mean_lst     =('mean_lst_celsius', 'mean'),
                    mean_rain    =('total_rain_mm', 'mean'),
                    mean_ndvi    =('mean_ndvi', 'mean'),
                    mean_fri     =('fire_risk_index', 'mean'),
                    fire_months  =('has_fire_event', 'sum')
                )
                .reset_index())

    annual['total_fire_ha'] = annual['total_fire_ha'].round(0).astype(int)
    annual['max_fire_ha']   = annual['max_fire_ha'].round(0).astype(int)
    annual['mean_lst']      = annual['mean_lst'].round(2)
    annual['mean_rain']     = annual['mean_rain'].round(1)
    annual['mean_ndvi']     = annual['mean_ndvi'].round(4)
    annual['mean_fri']      = annual['mean_fri'].round(4)

    print("=" * 70)
    print("📋 RINGKASAN KEBAKARAN TAHUNAN – PROVINSI RIAU (2015–2025)")
    print("=" * 70)
    print(annual.to_string(index=False))
    print()

    worst = annual.loc[annual['total_fire_ha'].idxmax()]
    print(f"🔥 Tahun terparah: {int(worst['year'])} "
          f"(total {worst['total_fire_ha']:,} ha, "
          f"{int(worst['fire_months'])} bulan kejadian signifikan)")

except NameError:
    print("⚠️  Jalankan sel sebelumnya (No.9) terlebih dahulu untuk memuat CSV.")


---
## 📝 Catatan & Referensi

| Variabel | Dataset | Resolusi | Keterangan |
|---|---|---|---|
| Burned Area | MODIS/061/MCD64A1 | 500m/bulanan | Area terbakar (burn date > 0) |
| Active Fire Terra | MODIS/061/MOD14A1 | 1km/harian | FireMask ≥ 8 (high confidence) |
| Active Fire Aqua | MODIS/061/MYD14A1 | 1km/harian | FireMask ≥ 8 (high confidence) |
| LST | MODIS/061/MOD11A2 | 1km/8-harian | Scale 0.02, offset −273.15 K |
| Curah Hujan | UCSB-CHG/CHIRPS/DAILY | ~5km/harian | Jumlah bulanan (mm) |
| NDVI | MODIS/061/MOD13A2 | 1km/16-harian | Scale 0.0001 |
| Kecepatan Angin | ECMWF/ERA5_LAND/MONTHLY_AGGR | ~9km/bulanan | √(u² + v²) |

**Fire Risk Index (FRI):**
```
FRI = LST_norm × 0.35 + (1 − NDVI_norm) × 0.35 + (1 − rain_norm) × 0.20 + wind_norm × 0.10
```
Kategori: Rendah (< 0.3) | Sedang (0.3–0.5) | Tinggi (0.5–0.7) | Sangat Tinggi (> 0.7)

**Referensi:**
- [GEE Catalog – MCD64A1](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD64A1)
- [geemap Documentation](https://geemap.org)
- [CHIRPS Rainfall](https://www.chc.ucsb.edu/data/chirps)
